# Constraining random forest predictions with prepared DGVM data

In [ ]:
# Libraries
import xarray as xr
import matplotlib.pyplot as plt

In [ ]:
# Directories
dir04 = '../paper_ecosystem_carbon/output/04_dgvm_prep/'
dir05 = '../paper_ecosystem_carbon/output/05_rf_constrain/'

---

In [ ]:
# List of ecosystem types
list_etype = ['trop_ef', 'trop_df', 'temp_ebf', 'temp_dbf', 'temp_enf',
              'temp_dnf', 'glob_shrub', 'glob_grass', 'glob_tundr', 
              'glob_mount', 'glob_crop', 'glob_past']

In [ ]:
# Get filled dgvm dataset
ds_dgvm = xr.open_zarr(f'{dir04}ds_dgvm_rffill_r025.zarr').compute()

In [ ]:
# Dataset with mean across models for each grid cell
ds_dgvm_mean = ds_dgvm.mean('model')
# Dataset with std across models for each grid cell
ds_dgvm_std = ds_dgvm.std('model')

# Dataset with dgvm mean plus std to be used as upper bound limit
ds_dgvm_upper = ds_dgvm_mean + ds_dgvm_std
# Dataset with dgvm minus plus std to be used as lower bound limit
ds_dgvm_lower = ds_dgvm_mean - ds_dgvm_std

# Ensure all values are at least zero
ds_dgvm_lower = xr.where(ds_dgvm_lower < 0, 0, ds_dgvm_lower)

In [ ]:
# For each variable, get the number of dgvms with prepared data 
df_ndgvm = ds_dgvm \
    .to_array(dim='v') \
    .notnull() \
    .any(dim=[d for d in ds_dgvm.dims if d != 'model']) \
    .sum(dim='model') \
    .rename('ndgvm') \
    .compute() \
    .to_dataframe() \
    .reset_index()

df_ndgvm

In [ ]:
def constrain_predictions():

    """Constrain predictions using upper and lower values from 
        mean +- std from prepared dgvm data and export as zarr"""
    
    # Constrain predicition arrays
    def constrain_estimates_da(ds, ctype, etype):
        """ Constrain predicition arrays using upper and lower values from 
        mean +- std from prepared dgvm data"""
 
        # Get variables
        da_rfpred = ds[f'{ctype}_{etype}']
        da_dgvm_upper = ds_dgvm_upper[f'{ctype}_{etype}']
        da_dgvm_lower = ds_dgvm_lower[f'{ctype}_{etype}']
        
        # Constrain random forest predictions with prepared dgvm mean +- std
        # Constrain only if at least 5 dgvm have prepared data
        if df_ndgvm[df_ndgvm.v == f'{ctype}_{etype}'].ndgvm.item() >= 5:
            da_best = xr.where(da_rfpred < da_dgvm_lower, da_dgvm_lower, da_rfpred)
            da_best = xr.where(da_best > da_dgvm_upper, da_dgvm_upper, da_best)
        else:
            da_best = da_rfpred
            print(f'Not constrained: {ctype}_{etype}')
        return da_best

    # Get regridded random forest datasets
    ds_rfpred = xr.open_mfdataset(f'{dir05}ds_rfpred_*_rcon_r025.zarr', 
                                  engine='zarr').persist()
    
    # Dataset for constrained prediction arrays
    ds_rfconstr = xr.Dataset()
    # Constrain predictions and add to dataset
    for ctype in ['cveg', 'csoil']:
        for etype in list_etype:
            da_rfconstr = constrain_estimates_da(ds_rfpred, ctype, etype)                
            ds_rfconstr[f'{ctype}_{etype}'] = da_rfconstr
                
    # Export
    ds_rfconstr.to_zarr(f'{dir05}ds_rfconstr_r025.zarr', mode='w');


constrain_predictions()